# Bring your own vocabulary: teaching an LLMON model a planning vocabulary — ScienceWorld

This notebook is the whole workflow for one use case, end to end:

1. take an agent's trajectories (here, **ScienceWorld**'s reference solutions),
2. define a **tag vocabulary** the model never saw in post-training,
3. render every step of every trajectory in that vocabulary,
4. teach the vocabulary to [`granite-4.1-3b-llmon-preview`](https://huggingface.co/ibm-granite/granite-4.1-3b-llmon-preview) with a small **LoRA**,
5. run the trained model as an agent inside ScienceWorld, and score it against the untrained base model.

**ScienceWorld** ([allenai/ScienceWorld](https://github.com/allenai/ScienceWorld), Apache-2.0) is a text world in which
an agent carries out multi-step science procedures: boil water, test whether an object conducts electricity, find the
melting point of a substance. Each game is scored 0–100 with partial credit. The agent loop follows the **ReAct
baseline released with SwiftSage** ([SwiftSage](https://github.com/SwiftSage/SwiftSage), `science_world` branch).



## Step 0 — Install

ScienceWorld runs a Java simulator, so it needs **Java 8 or newer** on the machine (`java -version`). 

In [1]:
%pip install -q "scienceworld==1.1.3" torch transformers peft trl datasets accelerate scikit-learn editdistance
%pip install -q -e ../../..
import sys, os, shutil, subprocess
# an editable install is picked up by Python only at start-up; add the repo root now so no kernel restart is needed
REPO_ROOT = os.path.abspath("../../..")
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("java:", subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()[0] if shutil.which("java") else "NOT FOUND - install a JDK (e.g. conda install -c conda-forge openjdk=11)")

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


java: openjdk version "11.0.30-internal" 2026-01-20


## Step 1 — Get SwiftSage (for its data and its action matcher)

SwiftSage's `science_world` branch holds ScienceWorld's reference solutions (`goldpaths-all.zip`), the example game
the untrained baseline is prompted with (`prompt.jsonl`), and the action matcher its ReAct baseline uses. The clone
is pinned to the commit used for the published run, and kept in `./work/`.

In [2]:
import os, sys, zipfile, subprocess
from pathlib import Path
WORK = Path("work"); WORK.mkdir(exist_ok=True)
SS = WORK / "SwiftSage"
SS_COMMIT = "b8ea624231a6a8d83d78c1c12bea1b44b8ea758b"
if not SS.exists():
    subprocess.run(["git", "clone", "-q", "-b", "science_world", "https://github.com/SwiftSage/SwiftSage.git", str(SS)], check=True)
subprocess.run(["git", "-C", str(SS), "checkout", "-q", SS_COMMIT], check=True)
GOLD = WORK / "goldsequences-all.json"
if not GOLD.exists():
    with zipfile.ZipFile(SS / "data_utils" / "goldpaths-all.zip") as z:
        name = z.namelist()[0]; z.extract(name, WORK); (WORK / name).rename(GOLD)
print("SwiftSage @", subprocess.run(["git", "-C", str(SS), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip(),
      "| reference solutions:", GOLD, f"({GOLD.stat().st_size / 1e6:.0f} MB)")

SwiftSage @ b8ea624 | reference solutions: work/goldsequences-all.json (491 MB)


## Step 2 — Environment check

In [3]:
import json, random, collections, time, types, re
import torch, transformers, peft, trl
from datasets import Dataset
from llmon.lang.json.llmon_converters import LLMONConverters
transformers.utils.logging.disable_progress_bar()          # keep the outputs readable
transformers.utils.logging.set_verbosity_error()
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| transformers", transformers.__version__,
      "| peft", peft.__version__, "| trl", trl.__version__)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

torch 2.14.1+cu130 | cuda True | transformers 5.21.0 | peft 0.21.2 | trl 1.16.0
gpu: NVIDIA H200


## Step 3 — Settings

`FULL_RUN = False` is the demo: 4 training games per task and 300 steps, then 9 test games.
`FULL_RUN = True` reproduces the published run: up to 40 games per task, one full epoch, and all 81 test games.

In [4]:
FULL_RUN = False
MODEL, REVISION = "ibm-granite/granite-4.1-3b-llmon-preview", "273198fbfd138f2a33506bb47fd7dfc4b2189d7a"
BASELINE = "ibm-granite/granite-4.1-3b"
OUT = WORK / ("full" if FULL_RUN else "demo")
MAX_PER_TASK = 40 if FULL_RUN else 4          # training games per task (one task has 692 games; cap so it can't dominate)
MAX_STEPS = -1 if FULL_RUN else 300           # -1 = one full epoch
HISTORY, MAX_LENGTH, OBS_CHARS, SEED = 10, 6144, 600, 0
EVAL_TASKS = list(range(3, 30)) if FULL_RUN else [5, 17, 25]     # SwiftSage evaluates tasks 3-29, 3 test games each
print("mode:", "FULL" if FULL_RUN else "DEMO", "| model", MODEL, "@", REVISION[:8])

mode: DEMO | model ibm-granite/granite-4.1-3b-llmon-preview @ 273198fb


## Step 4 — Look at the raw data: one trajectory

Solutions are stored per task and per game ("variation"), each with a `fold` of train / dev / test. **Only train-fold
solutions are used for training;** the test games are what we evaluate on. Each step has the action, what the game
replied, and the running score.

In [5]:
gold = json.load(open(GOLD))
seq = next(s for s in gold["0"]["goldActionSequences"] if s["fold"] == "train")
short = lambda s, n=OBS_CHARS: (lambda t: t if len(t) <= n else t[:n] + " ...")(" ".join(str(s).replace("\t", " ").split()))
print("task:", gold["0"]["taskName"], "| variation", seq["variationIdx"], "| steps", len(seq["path"]))
print("goal:", short(seq["taskDescription"], 200), "\n")
for st in seq["path"][1:7]:
    print(f"> {st['action']}\n  {short(st['observation'], 110)}   [score {st['score']}]")
n_train = {gold[t]["taskName"]: sum(s["fold"] == "train" for s in gold[t]["goldActionSequences"]) for t in sorted(gold, key=int)}
print(f"\n{len(n_train)} tasks, {sum(n_train.values())} training games in total")

task: boil | variation 0 | steps 40
goal: Your task is to boil water. For compounds without a boiling point, combusting the substance is also acceptable. First, focus on the substance. Then, take actions that will cause it to change its state ... 

> open door to kitchen
  The door is already open.   [score 0.0]
> go to kitchen
  You move to the kitchen.   [score 0.0]
> look around
  This room is called the kitchen. In it, you see: the agent a substance called air a chair. On the chair is: no ...   [score 0.0]
> pick up thermometer
  You move the thermometer to the inventory.   [score 0.0]
> open cupboard
  The cupboard is now open.   [score 0.0]
> pick up metal pot
  You move the metal pot to the inventory.   [score 0.0]

30 tasks, 3592 training games in total


## Step 5 — The vocabulary

The tags sit inside LLMON's channels, which `granite-4.1-3b-llmon-preview` was post-trained on. Everything below the
channel name is **new vocabulary**: the model has never seen these tags, and the LoRA teaches them.

| channel | tags | what they hold |
|---|---|---|
| `system` | (role sentence) | who the agent is |
| `instr` | `instr.task.goal` | the task, e.g. "boil water" (the only instruction) |
| `req` | `req.planning_policy.rule` ×3 | one action per step · focus on the named object first · don't repeat a failed action |
| `data` | `data.state.room`, `data.state.inventory` | where the agent is and what it holds |
| `data` | one block per recent step: `data.step`, `data.action`, `data.observation`, `data.score` | the last 10 steps |
| target | `assistant.next_action` | the action the model writes |

One training example is **one step**: the prompt is everything before that step, the completion is its action.
`render` below is the whole vocabulary: it builds the message, and the model's own chat template adds the `exec`
block that binds the instruction, rules and data.

In [6]:
conv = LLMONConverters(converting_to_json=False, with_mellea=False, keys_as_item=False,
                      no_expand_tags=False, no_openend_tagname=True, no_composite_tags=True)
llmonize = lambda obj: conv.json_dict_to_llmon_str(obj, compact=True)

SYSTEM = ("You are a science agent solving a task in a text environment. Given the goal, the current state and the "
          "recent steps, produce the next environment action.")
RULES = ["Give exactly one environment action, in the environment's own command form (e.g. go to kitchen, pick up thermometer, focus on water).",
         "Follow the task's instructions in order: focus on the object the task names before acting on it.",
         "Do not repeat an action that did not change anything; try a different step."]

def state_of(step):                      # where the agent is and what it holds, from a solution step
    return {"room": short(step.get("freelook", "")), "inventory": short(step.get("inventory", ""), 300)}

def render(tok, goal, state, history):
    data = [{"value": llmonize({"data": {"state": state}}), "exec": True}]
    for k, h in enumerate(history, 1):
        data.append({"value": llmonize({"data": {"step": k, "action": h["action"], "observation": h["observation"],
                                                 "score": h["score"]}}), "exec": True})
    msg = {"role": "user",
           "instruction": [{"value": llmonize({"instr": {"task": {"goal": goal}}}), "exec": True}],
           "requirement": [{"value": llmonize({"req": {"planning_policy": {"rule": r}}}), "exec": True} for r in RULES],
           "data": data}
    return tok.apply_chat_template([{"role": "system", "content": SYSTEM}, msg], tokenize=False, add_generation_prompt=True)

def target(action):
    return llmonize({"assistant": {"next_action": action}}) + "<|open_end|>"

NEXT = re.compile(r"<\|open\|>assistant<\|\.\|>next_action<\|close\|>(.*?)<\|open_end\|>", re.S)
def read_action(raw):                    # the model's output -> the action string
    m = NEXT.search(raw)
    return m.group(1).replace('\\"', '"').replace("\\n", " ").strip() if m else ""

tok = transformers.AutoTokenizer.from_pretrained(MODEL, revision=REVISION)
path, hist = seq["path"], []
for st in path[:3]:
    hist.append({"action": st["action"], "observation": short(st["observation"]), "score": float(st["score"])})
p, t = render(tok, short(seq["taskDescription"], 1000), state_of(path[2]), hist), target(path[3]["action"])
print("--- prompt (the model sees one line; broken here at each tag) ---")
print(p.replace("<|open|>", "\n<|open|>")[:2600], "\n ...")
print("\n--- target ---\n" + t)
print("\nread back:", repr(read_action(t)))

--- prompt (the model sees one line; broken here at each tag) ---

<|open|>system<|:|>Ðï<|close|>You are a science agent solving a task in a text environment. Given the goal, the current state and the recent steps, produce the next environment action.<|open_end|>
<|open|>instr<|:|>úî<|close|>
<|open|>instr<|.|>task<|.|>goal<|close|>Your task is to boil water. For compounds without a boiling point, combusting the substance is also acceptable. First, focus on the substance. Then, take actions that will cause it to change its state of matter.<|open_end|><|open_end|>
<|open|>data<|:|>íî<|close|>
<|open|>data<|.|>state<|.|>room<|close|>This room is called the hallway. In it, you see: the agent a substance called air a picture You also see: A door to the art studio (that is open) A door to the bedroom (that is open) A door to the greenhouse (that is open) A door to the kitchen (that is open) A door to the living room (that is open) A door to the workshop (that is open)<|open_end|>
<|open|>da

## Step 6 — Build the training set

Every step of every training trajectory becomes one example. Whole trajectories are held out for validation, and
examples longer than `MAX_LENGTH` tokens are dropped, never truncated.

In [7]:
rng = random.Random(SEED)
seqs = []
for tk in sorted(gold, key=int):
    tr = [s for s in gold[tk]["goldActionSequences"] if s["fold"] == "train"]
    rng.shuffle(tr)
    seqs += tr[:MAX_PER_TASK]
val_ids = set(rng.sample(range(len(seqs)), max(3, len(seqs) // 20)))
train, val = [], []
for si, s in enumerate(seqs):
    goal, path, hist = short(s["taskDescription"], 1000), s["path"], []
    for i, st in enumerate(path):
        ex = {"prompt": render(tok, goal, state_of(path[i - 1] if i else st), hist[-HISTORY:]),
              "completion": target(st["action"]) + tok.eos_token}
        if len(tok(ex["prompt"] + ex["completion"])["input_ids"]) <= MAX_LENGTH:
            (val if si in val_ids else train).append(ex)
        hist.append({"action": st["action"], "observation": short(st["observation"]), "score": float(st["score"])})
print(f"{len(seqs)} trajectories -> train {len(train)} examples, val {len(val)}")
ok = sum(read_action(e["completion"]) != "" for e in train + val)
print(f"targets that read back to an action: {ok}/{len(train) + len(val)}")

120 trajectories -> train 4326 examples, val 464
targets that read back to an action: 4790/4790


## Step 7 — LoRA and train

Rank-32 LoRA on all linear layers, loss on the completion only, best checkpoint by validation loss (the recipe of
`byo-vocab_LoRA_for_agents.ipynb`). About 62M trainable parameters, 1.8% of the model.

In [8]:
model = transformers.AutoModelForCausalLM.from_pretrained(MODEL, revision=REVISION, dtype=torch.bfloat16, device_map="auto")
lora = peft.LoraConfig(task_type="CAUSAL_LM", r=32, lora_alpha=64, lora_dropout=0.05, bias="none", target_modules="all-linear")
args = trl.SFTConfig(
    output_dir=str(OUT / "ckpts"), num_train_epochs=1, max_steps=MAX_STEPS, learning_rate=2e-4, lr_scheduler_type="cosine",
    per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=8, max_length=MAX_LENGTH,
    completion_only_loss=True, loss_type="nll", bf16=True, gradient_checkpointing=True,
    eval_strategy="steps", eval_steps=100, save_strategy="steps", save_steps=100, save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    logging_steps=25, disable_tqdm=True, report_to=[], seed=0)
trainer = trl.SFTTrainer(model=model, args=args, peft_config=lora, processing_class=tok,
                         train_dataset=Dataset.from_list(train), eval_dataset=Dataset.from_list(val))
trainer.model.print_trainable_parameters()
t0 = time.time(); trainer.train()
print(f"trained in {(time.time() - t0) / 60:.1f} min | validation loss by step:",
      [(h["step"], round(h["eval_loss"], 4)) for h in trainer.state.log_history if "eval_loss" in h])
trainer.model.save_pretrained(str(OUT / "adapter")); tok.save_pretrained(str(OUT / "adapter"))
print("saved", OUT / "adapter")


Adding EOS to train dataset:   0%|          | 0/4326 [00:00<?, ? examples/s]


Adding EOS to train dataset:  93%|█████████▎| 4009/4326 [00:00<00:00, 38248.78 examples/s]


Adding EOS to train dataset: 100%|██████████| 4326/4326 [00:00<00:00, 37894.64 examples/s]


Tokenizing train dataset:   0%|          | 0/4326 [00:00<?, ? examples/s]


Tokenizing train dataset:   1%|          | 29/4326 [00:00<00:15, 278.17 examples/s]


Tokenizing train dataset:   1%|▏         | 59/4326 [00:00<00:14, 284.51 examples/s]

... (204 outputs left out)

{'eval_loss': '0.32', 'eval_runtime': '24.6', 'eval_samples_per_second': '18.86', 'eval_steps_per_second': '18.86', 'eval_entropy': '0.7372', 'eval_mean_token_accuracy': '0.9806', 'eval_num_tokens': '2.939e+06', 'epoch': '0.5548'}


{'train_runtime': '945.3', 'train_samples_per_second': '2.539', 'train_steps_per_second': '0.317', 'train_loss': '0.5023', 'epoch': '0.5548'}


trained in 15.8 min | validation loss by step: [(100, 0.4012), (200, 0.3326), (300, 0.32)]


saved work/demo/adapter


## Step 8 — Sanity check: does it write the vocabulary?

On held-out steps the trained model should open `assistant.next_action` and write one action. (The demo model is
trained briefly, so expect some misses here.)

In [9]:
lm = trainer.model.eval()

def generate(m, t, prompt, max_new=40):
    ids = t(prompt, return_tensors="pt", add_special_tokens=False).to(m.device)
    with torch.no_grad():
        out = m.generate(**ids, max_new_tokens=max_new, do_sample=False, pad_token_id=t.eos_token_id)
    return t.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=False)

hits = 0
for e in val[:8]:
    want, got = read_action(e["completion"]), read_action(generate(lm, tok, e["prompt"]))
    hits += got == want
    print(f"gold: {want:34s} | model: {got:34s} {'OK' if got == want else ''}")
print(f"exact next action on {hits}/8 held-out steps")

gold: look around                        | model: look around                        OK


gold: open door to hallway               | model: open door to hallway               OK


gold: go to hallway                      | model: go to hallway                      OK


gold: open door to kitchen               | model: open door to kitchen               OK


gold: go to kitchen                      | model: go to kitchen                      OK


gold: look around                        | model: look around                        OK


gold: pick up thermometer                | model: pick up thermometer                OK


gold: open cupboard                      | model: open cupboard                      OK
exact next action on 8/8 held-out steps


## Step 9 — Play test games in ScienceWorld

Each turn: render the state in the vocabulary, generate, read `assistant.next_action`, and pass it to SwiftSage's
action matcher, which maps it to the closest valid command. The game's own score (0–100) is the result. Same loop as
SwiftSage's ReAct baseline: 100 game steps, and stop when the agent keeps repeating itself.

The matcher and the test-game selection are imported from the SwiftSage clone. Its module also imports libraries the
matcher doesn't use (T5, sentence-transformers, OpenAI), so those are stubbed out instead of installed.

In [10]:
for name in ("sentence_transformers", "slow_agent", "slow_agent.utils", "slow_agent.local_llm", "openai", "tiktoken"):
    sys.modules.setdefault(name, types.ModuleType(name))
sys.modules["sentence_transformers"].SentenceTransformer = object
sys.modules["slow_agent.utils"].completion_with_backoff = None
sys.path.insert(0, str(SS))
import eval_utils as EU                    # SwiftSage: findValidActionNew (action matcher), load_variation (test games)
from scienceworld import ScienceWorldEnv
DEMOS = json.load(open(SS / "baselines" / "ReAct_baseline" / "prompt.jsonl"))
quiet = types.SimpleNamespace(info=lambda *a, **k: None)
env = ScienceWorldEnv("", None, envStepLimit=100)
NAMES = env.getTaskNames()
flat = lambda s: s.replace("\n", " ").replace("\t", " ")

def play(task, variation, policy):
    env.load(NAMES[task], variation, "easy", generateGoldPath=True)
    desc = env.taskdescription()[18:]
    obs, info = env.reset()
    goal, hist, recent, score, last = short(desc, 1000), [], ["look around"], 0.0, 0.0
    state = {"room": short(obs), "inventory": short(env.inventory(), 300)}
    react = "\n\nHere is the task.\n" + flat(obs) + "\n" + desc + "\n>"      # transcript for the untrained baseline
    for step in range(200):
        a = policy(task, goal, state, hist, react)
        if a.startswith("think:"):
            obs, done = "OK.", False
        else:
            a = EU.findValidActionNew([a], env, info["look"], recent, None, quiet)
            obs, _, done, info = env.step(a)
            score = info["score"]
            if score < 0:                       # SwiftSage's rule: a failed game keeps its last score
                done, score = True, last
            last = score
            hist.append({"action": a, "observation": short(flat(obs)), "score": float(score)})
            state = {"room": short(info.get("look", "")), "inventory": short(env.inventory(), 300)}
        react += f" {a}\n{flat(obs)}\n>"; recent.append(a)
        if done or (len(recent) >= 5 and len(set(recent[-5:])) == 2):
            break
    return score

games = []
for tk in EVAL_TASKS:
    env.load(NAMES[tk], 0, "easy")
    games += [(tk, v) for v in EU.load_variation(env, {"set": "test_mini", "cut_off": True}, tk, quiet)]
print(f"{len(games)} test games across:", sorted({NAMES[t] for t, _ in games}))

9 test games across: ['find-animal', 'lifespan-longest-lived', 'power-component']


In [11]:
results = collections.defaultdict(dict)
llmon_policy = lambda task, goal, state, hist, react: read_action(generate(lm, tok, render(tok, goal, state, hist[-HISTORY:])))
for g in games:
    results["LLMON + LoRA"][g] = play(*g, llmon_policy)
del trainer, model, lm; torch.cuda.empty_cache()

# the baseline: untrained granite-4.1-3b with SwiftSage's own ReAct prompt (one example game of the same task)
gtok = transformers.AutoTokenizer.from_pretrained(BASELINE)
gm = transformers.AutoModelForCausalLM.from_pretrained(BASELINE, dtype=torch.bfloat16, device_map="auto").eval()
def granite_policy(task, goal, state, hist, react):
    raw = generate(gm, gtok, "Interact with a household to solve a task. Here is an example.\n" + DEMOS[str(task)] + react)
    lines = [l.strip() for l in raw.replace(gtok.eos_token, "").split("\n") if l.strip()]
    return lines[0].lstrip(">").strip() if lines else ""
for g in games:
    results["granite-4.1-3b untrained"][g] = play(*g, granite_policy)

cols = list(results)
print(f"{'game':44s}" + "".join(f"{c:>26s}" for c in cols))
for g in games:
    print(f"{NAMES[g[0]][:36]:36s} v{g[1]:<6d}" + "".join(f"{results[c][g]:>26.0f}" for c in cols))
print(f"{'MEAN SCORE':44s}" + "".join(f"{sum(results[c].values()) / len(games):>26.1f}" for c in cols))

game                                                      LLMON + LoRA  granite-4.1-3b untrained
find-animal                          v225                           25                        25
find-animal                          v226                           17                        17
find-animal                          v227                           25                        25
lifespan-longest-lived               v93                           100                        50
lifespan-longest-lived               v94                           100                       100
lifespan-longest-lived               v95                            50                        50
power-component                      v15                            63                        13
power-component                      v16                           100                         7
power-component                      v17                           100                         7
MEAN SCORE                    

## What this shows

From the full run (one epoch on 898 training trajectories, 42,950 examples; SwiftSage's 81 test games):

| setup | mean score | games at 100 |
|---|---|---|
| granite-4.1-3b-llmon-preview + LoRA, LLMON vocabulary | **47.1** | 22 |
| granite-4.1-3b, untrained | 24.7 | 6 |

A vocabulary the model had never seen was taught through a small adapter, and the model nearly doubled the untrained
baseline while matching an identically trained model given the same content as JSON.
